# 01 — Dataset Exploration & Visualisation
**Emotion-Aware E-Commerce Chatbot | MSc Dissertation | Heriot-Watt University**

All 4 training phases use `twcs.csv` — the Twitter Customer Support dataset (29k rows).

- **Phase 1** — Sentiment classification (positive / neutral / negative)
- **Phase 2** — High frustration examples (score ≥ 0.65)
- **Phase 3** — Moderate frustration examples (score 0.30–0.65)
- **Phase 4** — Full dataset (all customer messages)

In [ ]:
import sys
sys.path.insert(0, '..')

import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from data.dataset_loader import (
    load_phase1_sentiment,
    load_phase2_high_frustration,
    load_phase3_moderate_frustration,
    load_phase4_full,
    TWCS_PATH
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.family'] = 'Arial'
print('✅ Imports ready')

---
## Raw Dataset Overview

In [ ]:
# Load raw CSV
df_raw = pd.read_csv(TWCS_PATH)
print(f'Total rows:  {len(df_raw):,}')
print(f'Columns:     {list(df_raw.columns)}')

df_customer = df_raw[df_raw['inbound'] == True].copy()
df_agent    = df_raw[df_raw['inbound'] == False].copy()
print(f'Customer messages: {len(df_customer):,}')
print(f'Agent replies:     {len(df_agent):,}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(['Customer\n(inbound)', 'Agent\n(outbound)'],
            [len(df_customer), len(df_agent)],
            color=['#1565C0','#1B5E20'], edgecolor='white', width=0.5)
axes[0].set_title('Message Split — Customer vs Agent', fontweight='bold')
axes[0].set_ylabel('Count')
for i,v in enumerate([len(df_customer), len(df_agent)]):
    axes[0].text(i, v+100, f'{v:,}', ha='center', fontweight='bold')

# Message length
df_customer['length'] = df_customer['text'].str.len()
axes[1].hist(df_customer['length'], bins=40, color='#1565C0', edgecolor='white', alpha=0.8)
axes[1].set_title('Customer Message Length Distribution', fontweight='bold')
axes[1].set_xlabel('Character count')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('../evaluation/results/raw_dataset_overview.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Phase 1 — Sentiment Classification

In [ ]:
p1 = load_phase1_sentiment()
df1 = pd.DataFrame(p1)
label_names = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
df1['label_name'] = df1['label'].map(label_names)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
counts = df1['label_name'].value_counts()
colors = ['#ef5350','#bdbdbd','#66bb6a']
axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', width=0.5)
axes[0].set_title('Phase 1 — Sentiment Label Distribution (Balanced)', fontweight='bold')
axes[0].set_ylabel('Count')
for i,(l,v) in enumerate(counts.items()):
    axes[0].text(i, v+10, f'{v:,}', ha='center', fontweight='bold')

df1['length'] = df1['text'].str.len()
for label, color in zip(['Negative','Neutral','Positive'], colors):
    axes[1].hist(df1[df1['label_name']==label]['length'], bins=30, alpha=0.6, label=label, color=color)
axes[1].set_title('Message Length by Sentiment', fontweight='bold')
axes[1].set_xlabel('Character count')
axes[1].legend()

plt.tight_layout()
plt.savefig('../evaluation/results/phase1_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Total Phase 1 examples: {len(df1):,}')

---
## Phase 2 — High Frustration Examples

In [ ]:
p2 = load_phase2_high_frustration()
df2 = pd.DataFrame(p2)
print(f'Phase 2 examples: {len(df2):,}')
print(f'Score range: {df2["frustration_score"].min():.2f} – {df2["frustration_score"].max():.2f}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df2['frustration_score'], bins=20, color='#ef5350', edgecolor='white', alpha=0.85)
ax.axvline(0.78, color='red', linestyle='--', linewidth=2, label='Handover threshold (0.78)')
ax.set_title('Phase 2 — High Frustration Score Distribution', fontweight='bold')
ax.set_xlabel('Frustration Score')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig('../evaluation/results/phase2_high_frustration.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nSample high frustration messages:')
for _, row in df2.sample(4, random_state=1).iterrows():
    print(f'  [{row["frustration_score"]:.2f}] {row["text"][:100]}')

---
## Phase 3 — Moderate Frustration Examples

In [ ]:
p3 = load_phase3_moderate_frustration()
df3 = pd.DataFrame(p3)
print(f'Phase 3 examples: {len(df3):,}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df3['frustration_score'], bins=20, color='#ff9800', edgecolor='white', alpha=0.85)
ax.axvline(0.50, color='orange', linestyle='--', linewidth=2, label='Alert threshold (0.50)')
ax.set_title('Phase 3 — Moderate Frustration Score Distribution', fontweight='bold')
ax.set_xlabel('Frustration Score')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig('../evaluation/results/phase3_moderate_frustration.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Phase 4 — Full Dataset Summary

In [ ]:
p4 = load_phase4_full()
df4 = pd.DataFrame(p4)
print(f'Phase 4 total: {len(df4):,}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df4['frustration_score'], bins=25, color='#7B1FA2', edgecolor='white', alpha=0.85)
axes[0].axvline(0.50, color='orange', linestyle='--', linewidth=2, label='Alert (0.50)')
axes[0].axvline(0.78, color='red',    linestyle='--', linewidth=2, label='Handover (0.78)')
axes[0].set_title('Phase 4 — Full Score Distribution', fontweight='bold')
axes[0].set_xlabel('Frustration Score')
axes[0].set_ylabel('Count')
axes[0].legend()

# Phase comparison
phases  = ['Phase 1\nSentiment', 'Phase 2\nHigh Frust.', 'Phase 3\nModerate', 'Phase 4\nFull']
sizes   = [len(df1), len(df2), len(df3), len(df4)]
colors2 = ['#1565C0','#ef5350','#ff9800','#7B1FA2']
bars = axes[1].bar(phases, sizes, color=colors2, edgecolor='white', linewidth=1.5, width=0.55)
for bar, size in zip(bars, sizes):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+100,
                 f'{size:,}', ha='center', fontweight='bold', fontsize=11)
axes[1].set_title('Training Examples by Phase (all from twcs.csv)', fontweight='bold')
axes[1].set_ylabel('Examples')

plt.tight_layout()
plt.savefig('../evaluation/results/all_phases_summary.png', dpi=150, bbox_inches='tight')
plt.show()

summary = pd.DataFrame([
    {'Phase': 'Phase 1', 'Purpose': 'Sentiment classification',     'Examples': len(df1), 'Score range': 'Labels: Neg/Neu/Pos'},
    {'Phase': 'Phase 2', 'Purpose': 'High frustration training',    'Examples': len(df2), 'Score range': '0.65 – 0.95'},
    {'Phase': 'Phase 3', 'Purpose': 'Moderate frustration training','Examples': len(df3), 'Score range': '0.30 – 0.65'},
    {'Phase': 'Phase 4', 'Purpose': 'Full range fine-tuning',       'Examples': len(df4), 'Score range': '0.25 – 0.95'},
])
display(summary)
print(f'\nSource: twcs.csv — {len(df_raw):,} total rows')
print('✅ All charts saved to evaluation/results/')